In [0]:
# Setup inicial (idempotente): catálogo, schema e volumes
spark.sql("CREATE CATALOG IF NOT EXISTS main")
spark.sql("USE CATALOG main")
spark.sql("CREATE SCHEMA IF NOT EXISTS nyc_taxi")
spark.sql("USE nyc_taxi")
spark.sql("CREATE VOLUME IF NOT EXISTS bronze")
spark.sql("CREATE VOLUME IF NOT EXISTS silver")
spark.sql("CREATE VOLUME IF NOT EXISTS gold")
print("Catálogo/schema/volumes prontos: main.nyc_taxi (bronze, silver, gold).")

In [0]:
# Parâmetros principais (Solução 1: tabelas gerenciadas no Unity Catalog)

# CSV deve estar carregado no Volume bronze via UI:
# Catalog → main → nyc_taxi → Volumes → bronze → Upload
CSV_INPUT_PATH  = "/Volumes/main/nyc_taxi/bronze/yellow_tripdata_2016-02.csv"

CATALOG = "main"
SCHEMA  = "nyc_taxi"

TBL_BRONZE = f"{CATALOG}.{SCHEMA}.yellow_tripdata_2016_02_bronze"
TBL_SILVER = f"{CATALOG}.{SCHEMA}.yellow_tripdata_2016_02_silver"
TBL_GOLD   = f"{CATALOG}.{SCHEMA}.yellow_trip_2016_02_gold_daily"

print("Tabelas alvo:", TBL_BRONZE, TBL_SILVER, TBL_GOLD)

In [0]:
# Conferência: arquivo existe no Volume?
try:
    files = dbutils.fs.ls("/Volumes/main/nyc_taxi/bronze")
    display(files)
except Exception as e:
    print("Não consegui listar o volume. Verifique permissões do Unity Catalog.")
    raise e

if not any(f.name == "yellow_tripdata_2016-02.csv" for f in files):
    raise FileNotFoundError(
        "CSV não encontrado em /Volumes/main/nyc_taxi/bronze. "
        "Faça o upload em Catalog > main > nyc_taxi > Volumes > bronze."
    )

In [0]:
# =========================
# CAMADA BRONZE (ingestão)
# =========================

from pyspark.sql import functions as F

df_bronze_csv = (spark.read
                 .option("header", True)
                 .option("inferSchema", True)
                 .csv(CSV_INPUT_PATH))

# Nomes de colunas minúsculos/sem espaços
df_bronze = df_bronze_csv.toDF(*[c.strip().lower() for c in df_bronze_csv.columns])

display(df_bronze.limit(5))

# Salva como TABELA GERENCIADA (Delta) no Unity Catalog
spark.sql("USE CATALOG main")
spark.sql("USE nyc_taxi")

(df_bronze.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable(TBL_BRONZE))

print(f"Bronze gravada e registrada em {TBL_BRONZE}")

In [0]:
# =========================
# CAMADA SILVER (limpeza)
# =========================
# - Tipagem correta
# - Colunas derivadas
# - Janela temporal (fev/2016)
# - Regras simples de qualidade

from pyspark.sql import functions as F

dfb = spark.table(TBL_BRONZE)

# Helper: nomes que variam entre anos
def first_existing(candidates, cols):
    for c in candidates:
        if c in cols:
            return c
    return None

cols = dfb.columns
pickup_col  = first_existing(["tpep_pickup_datetime","pickup_datetime","lpep_pickup_datetime"], cols)
dropoff_col = first_existing(["tpep_dropoff_datetime","dropoff_datetime","lpep_dropoff_datetime"], cols)
dist_col    = first_existing(["trip_distance","distance"], cols)
pass_col    = first_existing(["passenger_count","passenger"], cols)
vendor_col  = first_existing(["vendorid","vendor_id"], cols)
pay_col     = first_existing(["payment_type","paymenttype"], cols)
fare_col    = first_existing(["fare_amount","fareamount"], cols)
tip_col     = first_existing(["tip_amount","tipamount"], cols)
total_col   = first_existing(["total_amount","totalamount"], cols)

dfs = dfb
if pickup_col:  dfs = dfs.withColumn("pickup_ts",  F.to_timestamp(F.col(pickup_col)))
if dropoff_col: dfs = dfs.withColumn("dropoff_ts", F.to_timestamp(F.col(dropoff_col)))
if dist_col:    dfs = dfs.withColumn("trip_distance",   F.col(dist_col).cast("double"))
if pass_col:    dfs = dfs.withColumn("passenger_count", F.col(pass_col).cast("int"))
if vendor_col:  dfs = dfs.withColumn("vendor_id",       F.col(vendor_col).cast("string"))
if pay_col:     dfs = dfs.withColumn("payment_type",    F.col(pay_col).cast("int"))
if fare_col:    dfs = dfs.withColumn("fare_amount",     F.col(fare_col).cast("double"))
if tip_col:     dfs = dfs.withColumn("tip_amount",      F.col(tip_col).cast("double"))
if total_col:   dfs = dfs.withColumn("total_amount",    F.col(total_col).cast("double"))

dfs = (dfs
       .withColumn("pickup_date", F.to_date("pickup_ts"))
       .withColumn("year",  F.year("pickup_ts"))
       .withColumn("month", F.month("pickup_ts"))
       .withColumn("day",   F.dayofmonth("pickup_ts"))
       .withColumn("hour",  F.hour("pickup_ts"))
       .withColumn("trip_minutes", (F.col("dropoff_ts").cast("long") - F.col("pickup_ts").cast("long"))/60.0)
       .withColumn("trip_hours",   F.col("trip_minutes")/60.0)
       .withColumn("fare_per_mile", F.when((F.col("trip_distance") > 0) & (F.col("fare_amount") >= 0),
                                           F.col("fare_amount")/F.col("trip_distance")))
       .withColumn("tip_rate", F.when((F.col("fare_amount") > 0) & (F.col("tip_amount") >= 0),
                                      F.col("tip_amount")/F.col("fare_amount")))
       .withColumn("avg_speed_mph", F.when((F.col("trip_hours") > 0) & (F.col("trip_distance") >= 0),
                                           F.col("trip_distance")/F.col("trip_hours")))
      )

# Filtro de qualidade + janela de fevereiro/2016
dfs_clean = (dfs
             .where("pickup_ts IS NOT NULL AND dropoff_ts IS NOT NULL")
             .where("pickup_date >= DATE('2016-02-01') AND pickup_date < DATE('2016-03-01')")
             .where("trip_distance > 0 AND trip_minutes > 0")
             .where("passenger_count IS NULL OR passenger_count >= 1"))

display(dfs_clean.select("pickup_ts","dropoff_ts","trip_distance","fare_amount","tip_amount","tip_rate","avg_speed_mph").limit(10))

# Salva como TABELA GERENCIADA particionada
(dfs_clean.write
 .format("delta")
 .mode("overwrite")
 .partitionBy("year","month")
 .saveAsTable(TBL_SILVER))

print(f"Silver gravada e registrada em {TBL_SILVER}")

In [0]:
# =======================
# CAMADA GOLD (agregação)
# =======================

from pyspark.sql import functions as F

dff = spark.table(TBL_SILVER)

gold_daily = (dff.groupBy(
                    "year","month","pickup_date","payment_type","vendor_id"
                )
                .agg(F.count("*").alias("n_trips"),
                     F.avg("trip_distance").alias("avg_miles"),
                     F.avg("trip_minutes").alias("avg_minutes"),
                     F.avg("fare_amount").alias("avg_fare"),
                     F.avg("tip_rate").alias("avg_tip_rate"),
                     F.avg("avg_speed_mph").alias("avg_speed_mph"))
                .orderBy("pickup_date","vendor_id","payment_type")
             )

display(gold_daily.limit(20))

(gold_daily.write
 .format("delta")
 .mode("overwrite")
 .partitionBy("year","month")
 .saveAsTable(TBL_GOLD))

print(f"Gold gravada e registrada em {TBL_GOLD}")

In [0]:
# =======================
# Consultas de validação
# =======================

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE {SCHEMA}")

print("Bronze (contagem):")
display(spark.sql(f"SELECT COUNT(*) AS rows FROM {TBL_BRONZE}"))

print("Silver (partições e estatísticas):")
q1 = f'''
SELECT year, month,
       COUNT(*) AS n_trips,
       AVG(trip_distance) AS avg_miles,
       AVG(fare_amount)   AS avg_fare,
       AVG(tip_rate)      AS avg_tip_rate
FROM {TBL_SILVER}
GROUP BY year, month
ORDER BY year, month
'''
display(spark.sql(q1))

print("Gold (top 10 dias por gorjeta média):")
q2 = f'''
SELECT pickup_date, vendor_id, payment_type, n_trips, avg_fare, avg_tip_rate
FROM {TBL_GOLD}
ORDER BY avg_tip_rate DESC
LIMIT 10
'''
display(spark.sql(q2))

In [0]:
# Leituras rápidas de volta (exemplos)

silver_all = spark.table(TBL_SILVER)
print("Silver rows:", silver_all.count())

silver_feb = silver_all.where("year = 2016 AND month = 2")
print("Silver rows (2016-02):", silver_feb.count())

gold_all = spark.table(TBL_GOLD)
print("Gold rows:", gold_all.count())